# 🔓 Solutions: Model Evaluation - Classification

> **Try first, peek second.** You learn far more by struggling with a problem for ten minutes than by reading its answer for ten seconds. Use these solutions to *compare* your approach, not to replace it. There is often more than one correct way.


In [1]:
import sys
sys.path.append("..")
from course_utils import *
setup()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score)

### 🧩 Problem 7.1: Count the four outcomes &nbsp; ⭐

Given the truth and predictions below (1 = positive), compute `tp`, `fp`, `fn`, `tn` **without using scikit-learn's confusion_matrix**.

In [2]:
y_true = np.array([1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1])
y_pred = np.array([1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1])

tp = int(((y_true == 1) & (y_pred == 1)).sum())
fp = int(((y_true == 0) & (y_pred == 1)).sum())
fn = int(((y_true == 1) & (y_pred == 0)).sum())
tn = int(((y_true == 0) & (y_pred == 0)).sum())
print(tp, fp, fn, tn, "→ sum", tp + fp + fn + tn)

4 1 2 5 → sum 12


In [3]:
# ✔ Self-check
check("TP = 4", lambda: tp == 4)
check("FP = 1", lambda: fp == 1)
check("FN = 2", lambda: fn == 2)
check("TN = 5", lambda: tn == 5)

✅ PASSED   TP = 4
✅ PASSED   FP = 1
✅ PASSED   FN = 2
✅ PASSED   TN = 5


True

**🧑‍🏫 Walkthrough.** Each outcome is one boolean condition on (truth, prediction). The four counts must add up to the number of samples (12): a good sanity check to always run.

### 🧩 Problem 7.2: From counts to metrics &nbsp; ⭐

Using the counts from the previous problem (given again below), compute: `accuracy`, `precision`, `recall`, `specificity`, `f1`.

In [4]:
tp, fp, fn, tn = 4, 1, 2, 5

accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
specificity = tn / (tn + fp)
f1 = 2 * precision * recall / (precision + recall)
print(f"accuracy {accuracy:.3f}, precision {precision:.3f}, recall {recall:.3f}, specificity {specificity:.3f}, F1 {f1:.3f}")

accuracy 0.750, precision 0.800, recall 0.667, specificity 0.833, F1 0.727


In [5]:
# ✔ Self-check
check("accuracy = 0.75", lambda: abs(accuracy - 0.75) < 1e-9)
check("precision = 0.8", lambda: abs(precision - 0.8) < 1e-9)
check("recall = 2/3", lambda: abs(recall - 2 / 3) < 1e-9)
check("specificity = 5/6", lambda: abs(specificity - 5 / 6) < 1e-9)
check("F1 ≈ 0.727", lambda: abs(f1 - 0.72727) < 1e-4)

✅ PASSED   accuracy = 0.75
✅ PASSED   precision = 0.8
✅ PASSED   recall = 2/3
✅ PASSED   specificity = 5/6
✅ PASSED   F1 ≈ 0.727


True

**🧑‍🏫 Walkthrough.** This model is *right 75% of the time* but finds only two-thirds of the positives. In a disease-screening context the 33% miss rate (1 − recall) would be the headline, not the 75% accuracy.

### 🧩 Problem 7.3: Beat the accuracy paradox &nbsp; ⭐⭐

Using the churn split provided (`X_train, X_test, y_train, y_test`):

1. Train `DummyClassifier(strategy="most_frequent")` → store test accuracy in `dummy_acc` and test recall in `dummy_recall`.
2. Train `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` → store its test `lr_acc`, `lr_recall`, `lr_f1`.
3. Store `True` or `False` in `accuracy_misleads`: *"does the dummy model have higher accuracy than **or equal to within 5 points** of the logistic model while having zero recall?"* (i.e. `dummy_acc >= lr_acc - 0.05 and dummy_recall == 0`).

In [6]:
churn = load("churn.csv")
X, y = churn.drop(columns="churned"), churn["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
dp = dummy.predict(X_test)
dummy_acc, dummy_recall = accuracy_score(y_test, dp), recall_score(y_test, dp)

lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
lp = lr.predict(X_test)
lr_acc, lr_recall, lr_f1 = accuracy_score(y_test, lp), recall_score(y_test, lp), f1_score(y_test, lp)

accuracy_misleads = bool(dummy_acc >= lr_acc - 0.05 and dummy_recall == 0)
print(f"dummy: acc {dummy_acc:.3f}, recall {dummy_recall:.2f}    logistic: acc {lr_acc:.3f}, recall {lr_recall:.2f}, F1 {lr_f1:.2f}")

dummy: acc 0.848, recall 0.00    logistic: acc 0.856, recall 0.11, F1 0.18


In [7]:
# ✔ Self-check
check("dummy accuracy is above 80%", lambda: dummy_acc > 0.80)
check("dummy recall is exactly 0", lambda: dummy_recall == 0)
check("logistic regression finds some churners (recall > 0)", lambda: lr_recall > 0)
check("F1 is much lower than accuracy for the logistic model (imbalance bites)", lambda: lr_f1 < lr_acc - 0.2)
check("accuracy_misleads is True", lambda: accuracy_misleads is True)

✅ PASSED   dummy accuracy is above 80%
✅ PASSED   dummy recall is exactly 0
✅ PASSED   logistic regression finds some churners (recall > 0)
✅ PASSED   F1 is much lower than accuracy for the logistic model (imbalance bites)
✅ PASSED   accuracy_misleads is True


True

**🧑‍🏫 Walkthrough.** The two models differ by only a few points of accuracy, yet one catches no churners at all. **Whenever accuracy looks great, ask "what would a model that always predicts the majority class score?"**: that number is your accuracy baseline.

### 🧩 Problem 7.4: Find the threshold for 80% recall &nbsp; ⭐⭐

The business needs to **catch at least 80% of churners**. Use the split and model provided (train / validation / test; probabilities for validation and test are already computed).

1. Over thresholds `np.arange(0.02, 0.96, 0.01)`, find the **largest** threshold whose **validation recall is ≥ 0.80** (a larger threshold flags fewer customers, so it gives the best precision). Store it in `best_threshold`.
2. Store the validation precision and recall at that threshold in `val_precision` and `val_recall`.
3. Apply the *same* threshold to the **test** probabilities and store `test_recall`, `test_precision`.

In [8]:
churn = load("churn.csv")
X, y = churn.drop(columns="churned"), churn["churned"]
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=0, stratify=y_rest)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
p_val, p_test = model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]
thresholds = np.arange(0.02, 0.96, 0.01)

best_threshold = max(t for t in thresholds if recall_score(y_val, (p_val >= t).astype(int)) >= 0.80)
flag_val, flag_test = (p_val >= best_threshold).astype(int), (p_test >= best_threshold).astype(int)
val_precision, val_recall = precision_score(y_val, flag_val), recall_score(y_val, flag_val)
test_precision, test_recall = precision_score(y_test, flag_test), recall_score(y_test, flag_test)
print(f"threshold {best_threshold:.2f}: validation P={val_precision:.2f} R={val_recall:.2f} | test P={test_precision:.2f} R={test_recall:.2f}")

threshold 0.15: validation P=0.31 R=0.81 | test P=0.34 R=0.84


In [9]:
# ✔ Self-check
_ok = [t for t in thresholds if recall_score(y_val, (p_val >= t).astype(int)) >= 0.80]
check("best_threshold is the largest one with validation recall ≥ 0.80", lambda: abs(best_threshold - max(_ok)) < 1e-9)
check("validation recall ≥ 0.80", lambda: val_recall >= 0.80)
check("test recall is in the same region (> 0.65)", lambda: test_recall > 0.65)
check("precision is modest (the price of high recall): < 0.5", lambda: val_precision < 0.5)

✅ PASSED   best_threshold is the largest one with validation recall ≥ 0.80
✅ PASSED   validation recall ≥ 0.80
✅ PASSED   test recall is in the same region (> 0.65)
✅ PASSED   precision is modest (the price of high recall): < 0.5


True

**🧑‍🏫 Walkthrough.** To catch ~80% of churners we must accept a low precision: most flagged customers won't actually leave. Whether that is acceptable is a *business* question (what does a retention offer cost?). Note that the test recall differs a little from the validation recall: the threshold was tuned on one sample and applied to another. Expect that wobble; it's the honest estimate.

### 🧩 Problem 7.5: AUC by counting pairs &nbsp; ⭐⭐

Implement `auc_by_pairs(labels, scores)`: for every (positive, negative) pair count `1` if the positive's score is higher, `0.5` if tied, `0` otherwise, and return the **average**.

In [10]:
def auc_by_pairs(labels, scores):
    labels, scores = np.asarray(labels), np.asarray(scores)
    pos, neg = scores[labels == 1], scores[labels == 0]
    return float(np.mean([(p > n) + 0.5 * (p == n) for p in pos for n in neg]))

_rng = np.random.default_rng(0)
_labels = _rng.integers(0, 2, 200); _scores = _rng.normal(size=200) + _labels
print(auc_by_pairs(_labels, _scores), roc_auc_score(_labels, _scores))

0.7573641056787124 0.7573641056787124


In [11]:
# ✔ Self-check
_rng = np.random.default_rng(0)
_labels = _rng.integers(0, 2, 200); _scores = _rng.normal(size=200) + _labels
check("matches roc_auc_score on 200 random points", lambda: abs(auc_by_pairs(_labels, _scores) - roc_auc_score(_labels, _scores)) < 1e-9)
check("handles ties (AUC of constant scores = 0.5)", lambda: abs(auc_by_pairs(np.array([0, 1, 0, 1]), np.array([.5, .5, .5, .5])) - 0.5) < 1e-9)

✅ PASSED   matches roc_auc_score on 200 random points
✅ PASSED   handles ties (AUC of constant scores = 0.5)


True

**🧑‍🏫 Walkthrough.** AUC is *exactly* the fraction of correctly ranked positive–negative pairs. (sklearn computes it much faster by sorting, because the double loop is O(n²).) A constant score has AUC 0.5 (pure guessing) which is a good sanity check for any ranking metric.

### 🧩 Problem 7.6: Pick the best model by AUC &nbsp; ⭐⭐

Using the churn split provided, train these three models and compute each one's **test ROC-AUC** (from `predict_proba(...)[:, 1]`). Store in a dictionary `aucs` keyed `"logistic"`, `"tree"`, `"knn"`, and store the winner's key in `best_name`.

* `"logistic"` → `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))`
* `"tree"` → `DecisionTreeClassifier(max_depth=3, random_state=0)`
* `"knn"` → `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))`

In [12]:
churn = load("churn.csv")
X, y = churn.drop(columns="churned"), churn["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

models = {"logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
          "tree": DecisionTreeClassifier(max_depth=3, random_state=0),
          "knn": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))}
aucs = {n: roc_auc_score(y_test, m.fit(X_train, y_train).predict_proba(X_test)[:, 1]) for n, m in models.items()}
best_name = max(aucs, key=aucs.get)
print({k: round(v, 3) for k, v in aucs.items()}, "→", best_name)

{'logistic': 0.848, 'tree': 0.733, 'knn': 0.775} → logistic


In [13]:
# ✔ Self-check
check("three AUCs recorded", lambda: sorted(aucs) == ["knn", "logistic", "tree"])
check("all better than random (> 0.5)", lambda: all(v > 0.5 for v in aucs.values()))
check("best_name is the model with the highest AUC", lambda: best_name == max(aucs, key=aucs.get))
check("logistic regression ranks well (AUC > 0.8)", lambda: aucs["logistic"] > 0.8)

✅ PASSED   three AUCs recorded
✅ PASSED   all better than random (> 0.5)
✅ PASSED   best_name is the model with the highest AUC
✅ PASSED   logistic regression ranks well (AUC > 0.8)


True

**🧑‍🏫 Walkthrough.** Logistic regression wins: churn here is generated by a (noisy) log-odds formula, so a model whose *form* matches that (a linear score passed through a sigmoid) has the right inductive bias. ROC-AUC lets us compare the three models without committing to any threshold.

### 🧩 Problem 7.7: Cost-sensitive threshold &nbsp; ⭐⭐⭐

A missed churner costs **\$500**; a false alarm costs **\$50**. With the validation probabilities provided:

1. Write `cost(y_true, p, t)` returning the total cost when flagging customers with `p >= t`.
2. Search `np.arange(0.02, 0.96, 0.01)` for the threshold minimising the **validation** cost → `best_t`.
3. Store the validation cost at `best_t` in `cost_best` and at the default 0.5 in `cost_default`.

In [14]:
churn = load("churn.csv")
X, y = churn.drop(columns="churned"), churn["churned"]
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=0, stratify=y_rest)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
p_val = model.predict_proba(X_val)[:, 1]
thresholds = np.arange(0.02, 0.96, 0.01)

def cost(y_true, p, t):
    fn = ((y_true == 1) & (p < t)).sum()
    fp = ((y_true == 0) & (p >= t)).sum()
    return 500 * fn + 50 * fp

costs = [cost(y_val.to_numpy(), p_val, t) for t in thresholds]
best_t = float(thresholds[int(np.argmin(costs))])
cost_best, cost_default = cost(y_val.to_numpy(), p_val, best_t), cost(y_val.to_numpy(), p_val, 0.5)
print(f"best threshold {best_t:.2f}: cost ${cost_best:,} vs ${cost_default:,} at 0.5")

best threshold 0.08: cost $6,350 vs $18,550 at 0.5


In [15]:
# ✔ Self-check
check("cost function works on a tiny example (1 FN + 1 FP = 550)", lambda: cost(np.array([1, 0]), np.array([0.2, 0.9]), 0.5) == 550)
check("cost_default is the cost at t = 0.5", lambda: cost_default == cost(y_val.to_numpy(), p_val, 0.5))
check("best_t is cheaper than the default", lambda: cost_best < cost_default)
check("best_t is well below 0.5 (misses cost 10× more)", lambda: best_t < 0.3)

✅ PASSED   cost function works on a tiny example (1 FN + 1 FP = 550)
✅ PASSED   cost_default is the cost at t = 0.5
✅ PASSED   best_t is cheaper than the default
✅ PASSED   best_t is well below 0.5 (misses cost 10× more)


True

**🧑‍🏫 Walkthrough.** When a miss costs ten times a false alarm, the cheapest policy flags many more customers than the default 0.5 rule would; the optimum lands close to the theoretical 50/(50+500) ≈ 0.09. The ML model hasn't changed: only the **decision rule** did. Always ask the business for the costs.